# CR1 — Cascade R-CNN 16-band (Colab, competition-download)

Uses the **competition** download (allowed from Colab) + an **embedded split** — never touches a private dataset.

**Steps:** (1) T4 GPU. (2) Run cell 1 → it restarts. (3) Runtime > Run all; upload `kaggle.json` when asked.

If the competition download 403s too, it's a hard Colab IP block and we use Kaggle instead.

In [ ]:
# === STEP 1: pin working kaggle client, then RESTART so it loads ===
import subprocess, sys, importlib.metadata, os
try: v = importlib.metadata.version("kaggle")
except Exception: v = None
if v != "2.1.2":
    subprocess.run([sys.executable,"-m","pip","install","-q","--force-reinstall","kaggle==2.1.2"], check=True)
    print("Installed kaggle==2.1.2 — RESTARTING. After restart: Runtime > Run all again.")
    os.kill(os.getpid(), 9)
else:
    print("kaggle 2.1.2 ready — continue.")


In [ ]:
# === STEP 2: data via COMPETITION download (not the private dataset) + embedded split ===
import os, sys, subprocess, base64
from pathlib import Path
from google.colab import files
print(">>> Upload ~/Downloads/kaggle.json:")
up = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
src = "kaggle.json" if os.path.exists("kaggle.json") else list(up)[0]
os.replace(src, "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
import json as _j; _c=_j.load(open("/root/.kaggle/kaggle.json"))
os.environ["KAGGLE_USERNAME"]=_c["username"]; os.environ["KAGGLE_KEY"]=_c["key"]; print("auth as:", _c["username"])
BASE = "/content/kinput"
for d in (BASE, "/kaggle/working", "/kaggle/temp"): os.makedirs(d, exist_ok=True)
# 1) competition data -> BASE/hyperspectral-2026/{data_train,data_test,class.txt}
dest = f"{BASE}/hyperspectral-2026"; os.makedirs(dest, exist_ok=True)
print(">>> downloading competition data (this is the big one) ...")
r = subprocess.run(["kaggle","competitions","download","-c","hyperspectral-object-detection-challenge-2026","-p",dest], capture_output=True, text=True)
if r.stdout.strip(): print("STDOUT:", r.stdout[-1500:])
if r.stderr.strip(): print("STDERR:", r.stderr[-3000:])
if r.returncode != 0: raise SystemExit(f"competition download FAILED (exit {r.returncode}). Send me the STDERR above.")
import zipfile, glob
zips = glob.glob(f"{dest}/*.zip")
print(">>> unzipping", len(zips), "archive(s) ...")
for z in zips:
    with zipfile.ZipFile(z) as zf: zf.extractall(dest)
    os.remove(z)
# 2) embedded split_manifest.json (your own; no private dataset needed)
SPLIT_B64 = "..."
spd = f"{BASE}/plan_c/hsi_plan_c_ratio"; os.makedirs(spd, exist_ok=True)
open(f"{spd}/split_manifest.json","wb").write(base64.b64decode(SPLIT_B64))
# sanity
print("hyperspectral-2026 found:", len(list(Path(BASE).rglob("hyperspectral-2026"))), "(want >=1)")
print("split found          :", len(list(Path(BASE).rglob("hsi_plan_c_ratio/split_manifest.json"))), "(want 1)")
print("class.txt found      :", (Path(dest)/"class.txt").exists())
print(">>> bootstrap done. Now Runtime > Run all.")


# cr1-cascade-rcnn-f0

Cascade R-CNN (ResNet-50 FPN), 16-band, 18-class, fixed-fold 10-epoch architecture screen via MMDetection. Targets the mAP50->mAP75 localization gap (progressive box refinement). No competition submission.

In [ ]:
# CR1 = Cascade R-CNN R50-FPN, 16-band, 18-class, fixed-fold 10-epoch screen (MMDetection). No submission.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "openmim"], check=True)
subprocess.run([sys.executable, "-m", "mim", "install", "mmengine"], check=True)
subprocess.run([sys.executable, "-m", "mim", "install", "mmcv>=2.0.0,<2.2.0"], check=True)
subprocess.run([sys.executable, "-m", "mim", "install", "mmdet>=3.0.0,<3.4.0"], check=True)
from pathlib import Path
repo = Path("/kaggle/working/repos/mmdetection")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/open-mmlab/mmdetection.git",str(repo)], check=True)
import mmdet, mmcv, mmengine, torch
print("torch", torch.__version__, "| mmdet", mmdet.__version__, "| mmcv", mmcv.__version__)


In [ ]:
from pathlib import Path
Path('/kaggle/working/hsi_runner.py').write_text('"""Reproducible 16-band hyperspectral detector experiments and submission builder.\n\nSimple explanation:\n\nThis script:\n1. Reads the 16-band hyperspectral images.\n2. Reads the correct object boxes from XML files.\n3. Keeps some training images hidden for validation and holdout testing.\n4. Converts the data into the format YOLO expects.\n5. Trains one YOLO26m or RT-DETR-L object-detection model.\n6. Uses that model on the competition test images.\n7. Creates submission.csv only when holdout mAP50-95 is at least 0.50.\n\nImportant terms:\n- YOLO26m: an object detector. It predicts class, confidence, and box.\n- 16 bands: the 16 spectral channels in each competition image.\n- Validation: hidden training images used to check model progress.\n- Holdout: another hidden local set used as a safer final check.\n- Confidence: how sure the model is about one prediction.\n- Bounding box: rectangle around an object.\n\nThis version can read the competition raw PNG mosaics directly and applies the organizer-provided X2Cube conversion internally. It supports controlled aspect-ratio, cross-validation, spectral-augmentation, and same-checkpoint TTA experiments.\n"""\n\nfrom pathlib import Path\nimport argparse\nimport json\nimport random\nimport hashlib\nimport warnings\nimport importlib.metadata\nimport xml.etree.ElementTree as ET\n\nimport cv2\nfrom PIL import Image\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\n\n\n\ndef X2Cube(img, cellSize=4):\n    if img.ndim != 2 or any(n % cellSize for n in img.shape):\n        raise ValueError("Raw image sides must be divisible by 4")\n    B = [cellSize, cellSize]\n    skip = [cellSize, cellSize]\n    M, N = img.shape\n    col_extent = N - B[1] + 1\n    row_extent = M - B[0] + 1\n    start_idx = np.arange(B[0])[:, None] * N + np.arange(B[1])\n    didx = M * N * np.arange(1)\n    start_idx = (didx[:, None] + start_idx.ravel()).reshape((-1, B[0], B[1]))\n    offset_idx = np.arange(row_extent)[:, None] * N + np.arange(col_extent)\n    out = np.take(\n        img,\n        start_idx.ravel()[:, None] + offset_idx[::skip[0], ::skip[1]].ravel(),\n    )\n    out = np.transpose(out)\n    return out.reshape(M // cellSize, N // cellSize, cellSize * cellSize)\n\n\ndef read_classes(path):\n    names = [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]\n    if len(names) != 18:\n        raise ValueError(f"Expected 18 classes, found {len(names)}")\n    return names\n\n\ndef read_cube(path):\n    path = Path(path)\n    if path.suffix.lower() == ".png":\n        raw = np.array(Image.open(path))\n        if raw.ndim != 2:\n            raise ValueError(f"{path} must be a 2D raw mosaic PNG, got {raw.shape}")\n        x = X2Cube(raw)\n    elif path.suffix.lower() == ".npy":\n        x = np.load(path)\n    elif path.suffix.lower() == ".npz":\n        z = np.load(path)\n        x = z[z.files[0]]\n    elif path.suffix.lower() in {".tif", ".tiff"}:\n        ok, frames = cv2.imreadmulti(str(path), flags=cv2.IMREAD_UNCHANGED)\n        if not ok:\n            raise ValueError(f"Cannot read {path}")\n        x = np.stack(frames, axis=-1)\n    else:\n        raise ValueError(f"Unsupported cube format: {path.suffix}")\n    x = np.asarray(x)\n    if not np.isfinite(x).all():\n        raise ValueError(f"Non-finite image values in {path}")\n    if x.ndim != 3:\n        raise ValueError(f"{path} must be 3D, got {x.shape}")\n    if x.shape[-1] == 16:\n        return x.astype(np.float32)\n    if x.shape[0] == 16:\n        return np.moveaxis(x, 0, -1).astype(np.float32)\n    raise ValueError(f"{path} does not contain 16 bands: {x.shape}")\n\n\ndef collect_cubes(root):\n    root = Path(root)\n    files = []\n    for ext in ("*.png", "*.npy", "*.npz", "*.tif", "*.tiff"):\n        files.extend(root.rglob(ext))\n    files = sorted({p.resolve() for p in files})\n    if not files:\n        raise FileNotFoundError(\n            "No 16-band cubes found. Run the organizer-compatible X2Cube conversion first."\n        )\n    return files\n\n\ndef band_limits(files, samples_per_image=256, seed=42):\n    rng = np.random.default_rng(seed)\n    chunks = []\n    for p in files:\n        x = read_cube(p).reshape(-1, 16)\n        n = min(samples_per_image, len(x))\n        idx = rng.choice(len(x), n, replace=False)\n        chunks.append(x[idx])\n    sample = np.concatenate(chunks, axis=0)\n    lo = np.percentile(sample, 0.5, axis=0)\n    hi = np.percentile(sample, 99.5, axis=0)\n    hi = np.maximum(hi, lo + 1e-6)\n    return lo, hi\n\n\ndef model_geometry(height, width, input_policy):\n    if input_policy == "native":\n        return {\n            "original_height": height,\n            "original_width": width,\n            "model_height": height,\n            "model_width": width,\n            "pad_top": 0,\n            "pad_left": 0,\n        }\n    if input_policy != "square_pad":\n        raise ValueError(f"Unknown input policy: {input_policy}")\n    side = max(height, width)\n    return {\n        "original_height": height,\n        "original_width": width,\n        "model_height": side,\n        "model_width": side,\n        "pad_top": (side - height) // 2,\n        "pad_left": (side - width) // 2,\n    }\n\n\ndef resolve_input_policy(family, input_policy):\n    if input_policy != "auto":\n        return input_policy\n    return "square_pad" if family == "rtdetr" else "native"\n\n\ndef prepare_cube(cube, lo, hi, input_policy, pipeline_state=None):\n    if pipeline_state is None:\n        x = np.clip((cube - lo) / (hi - lo), 0, 1)\n        x = np.rint(x * 255).astype(np.uint8)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import apply_pipeline\n        except ModuleNotFoundError:\n            from pipeline import apply_pipeline\n        x = apply_pipeline(cube, pipeline_state, output="uint8")\n    height, width = x.shape[:2]\n    geometry = model_geometry(height, width, input_policy)\n    if input_policy == "square_pad":\n        padded = np.zeros(\n            (geometry["model_height"], geometry["model_width"], 16),\n            dtype=np.uint8,\n        )\n        top = geometry["pad_top"]\n        left = geometry["pad_left"]\n        padded[top : top + height, left : left + width] = x\n        x = padded\n    return x, geometry\n\n\ndef write_tiff(cube, path, lo, hi, input_policy, pipeline_state=None):\n    x, geometry = prepare_cube(cube, lo, hi, input_policy, pipeline_state)\n    chw = np.moveaxis(x, -1, 0)\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    ok = cv2.imwritemulti(str(path), chw)\n    if not ok:\n        raise IOError(f"Failed to write {path}")\n    return geometry\n\n\ndef transform_rows(rows, geometry):\n    source_width = geometry["original_width"]\n    source_height = geometry["original_height"]\n    target_width = geometry["model_width"]\n    target_height = geometry["model_height"]\n    pad_left = geometry["pad_left"]\n    pad_top = geometry["pad_top"]\n    transformed = []\n    for cls, cx, cy, bw, bh in rows:\n        transformed.append(\n            (\n                cls,\n                (cx * source_width + pad_left) / target_width,\n                (cy * source_height + pad_top) / target_height,\n                bw * source_width / target_width,\n                bh * source_height / target_height,\n            )\n        )\n    return transformed\n\n\ndef restore_boxes(boxes, geometry):\n    boxes = np.asarray(boxes, dtype=np.float32).copy()\n    boxes[:, [0, 2]] -= geometry["pad_left"]\n    boxes[:, [1, 3]] -= geometry["pad_top"]\n    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, geometry["original_width"])\n    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, geometry["original_height"])\n    return boxes\n\n\ndef spectral_augment(cube, rng, gain_limit, tilt_limit, noise_std):\n    cube = cube.astype(np.float32, copy=True)\n    gain = rng.uniform(1.0 - gain_limit, 1.0 + gain_limit)\n    tilt = rng.uniform(-tilt_limit, tilt_limit)\n    wavelength_axis = np.linspace(-1.0, 1.0, 16, dtype=np.float32)\n    cube *= gain * (1.0 + tilt * wavelength_axis[None, None, :])\n    if noise_std > 0:\n        scale = np.maximum(cube.std(axis=(0, 1)), 1e-6)\n        noise = rng.normal(0.0, noise_std, size=cube.shape).astype(np.float32)\n        cube += noise * scale[None, None, :]\n    return cube\n\n\ndef stable_rng(seed, stem, copy_index):\n    digest = hashlib.sha256(f"{seed}:{stem}:{copy_index}".encode()).digest()\n    return np.random.default_rng(int.from_bytes(digest[:8], "little"))\n\n\ndef expand_rgb_stem(target_model, source_model):\n    import torch\n\n    target_layers = [layer for layer in target_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    source_layers = [layer for layer in source_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    target = next((layer for layer in target_layers if layer.in_channels == 16), None)\n    source = next(\n        (\n            layer\n            for layer in source_layers\n            if layer.in_channels == 3\n            and target is not None\n            and layer.out_channels == target.out_channels\n            and layer.kernel_size == target.kernel_size\n        ),\n        None,\n    )\n    if target is None or source is None:\n        raise RuntimeError("Could not identify compatible 3-channel and 16-channel input stems")\n    expanded = source.weight.detach().mean(dim=1, keepdim=True).repeat(1, 16, 1, 1)\n    expanded *= 3.0 / 16.0\n    target.weight.data.copy_(expanded.to(device=target.weight.device, dtype=target.weight.dtype))\n    if target.bias is not None and source.bias is not None:\n        target.bias.data.copy_(source.bias.detach().to(device=target.bias.device, dtype=target.bias.dtype))\n    print("Initialized the 16-channel input stem from RGB weights with activation-scale correction")\n\n\ndef trainer_class(args):\n    if args.stem_init == "random":\n        return None\n    if args.family == "rtdetr":\n        from ultralytics.models.rtdetr.train import RTDETRTrainer\n\n        class HSITrainer(RTDETRTrainer):\n            def get_model(self, cfg=None, weights=None, verbose=True):\n                model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n                if weights is None:\n                    raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n                expand_rgb_stem(model, weights)\n                return model\n\n        return HSITrainer\n    from ultralytics.models.yolo.detect import DetectionTrainer\n\n    class HSITrainer(DetectionTrainer):\n        def get_model(self, cfg=None, weights=None, verbose=True):\n            model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n            if weights is None:\n                raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n            expand_rgb_stem(model, weights)\n            return model\n\n    return HSITrainer\n\n\ndef parse_voc(xml_path, class_to_id):\n    root = ET.parse(xml_path).getroot()\n    size = root.find("size")\n    width = int(float(size.findtext("width")))\n    height = int(float(size.findtext("height")))\n    rows = []\n    for obj in root.findall("object"):\n        name = obj.findtext("name").strip()\n        if name not in class_to_id:\n            raise ValueError(f"Unknown class {name} in {xml_path}")\n        b = obj.find("bndbox")\n        x1 = float(b.findtext("xmin"))\n        y1 = float(b.findtext("ymin"))\n        x2 = float(b.findtext("xmax"))\n        y2 = float(b.findtext("ymax"))\n        x1 = min(max(x1, 0.0), width)\n        x2 = min(max(x2, 0.0), width)\n        y1 = min(max(y1, 0.0), height)\n        y2 = min(max(y2, 0.0), height)\n        if x2 <= x1 or y2 <= y1:\n            continue\n        cx = ((x1 + x2) / 2) / width\n        cy = ((y1 + y2) / 2) / height\n        bw = (x2 - x1) / width\n        bh = (y2 - y1) / height\n        rows.append((class_to_id[name], cx, cy, bw, bh))\n    return width, height, rows\n\n\ndef annotation_map(xml_root):\n    xmls = list(Path(xml_root).rglob("*.xml"))\n    if not xmls:\n        raise FileNotFoundError(f"No VOC XML files found under {xml_root}")\n    return unique_stems(xmls)\n\n\ndef rarest_image_label(stems, boxes_by_stem):\n    counts = {}\n    for stem in stems:\n        for row in boxes_by_stem[stem]:\n            cls = int(row[0])\n            counts[cls] = counts.get(cls, 0) + 1\n    labels = []\n    for stem in stems:\n        classes = {int(row[0]) for row in boxes_by_stem[stem]}\n        if not classes:\n            labels.append(-1)\n        else:\n            labels.append(min(classes, key=lambda c: counts[c]))\n    return labels\n\n\ndef safe_split(stems, boxes_by_stem, seed):\n    labels = rarest_image_label(stems, boxes_by_stem)\n    try:\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            stratify=labels,\n        )\n        temp_labels = rarest_image_label(temp, boxes_by_stem)\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            stratify=temp_labels,\n        )\n    except ValueError as exc:\n        warnings.warn(f"Class-balanced split failed; using random split: {exc}")\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            shuffle=True,\n        )\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            shuffle=True,\n        )\n    return sorted(train), sorted(val), sorted(holdout)\n\n\ndef read_groups(path, stems):\n    if not path:\n        return {stem: stem for stem in stems}, "image_id_only"\n    frame = pd.read_csv(path, dtype=str)\n    expected = {"image_id", "group_id"}\n    if set(frame.columns) != expected:\n        raise ValueError(f"Groups file columns must be {sorted(expected)}")\n    if frame["image_id"].duplicated().any():\n        raise ValueError("Groups file contains duplicate image_id values")\n    groups = dict(zip(frame["image_id"], frame["group_id"]))\n    missing = sorted(set(stems) - set(groups))\n    if missing:\n        raise ValueError(f"Groups file is missing {len(missing)} training IDs")\n    return {stem: groups[stem] for stem in stems}, str(Path(path))\n\n\ndef cv_split(stems, boxes_by_stem, groups_by_stem, seed, folds, fold_index):\n    if folds < 2:\n        raise ValueError("cv_folds must be at least 2")\n    if not 0 <= fold_index < folds:\n        raise ValueError(f"cv_fold_index must be in 0..{folds - 1}")\n    labels = np.asarray(rarest_image_label(stems, boxes_by_stem))\n    groups = np.asarray([groups_by_stem[stem] for stem in stems])\n    holdout_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)\n    design_idx, holdout_idx = next(holdout_splitter.split(stems, labels, groups))\n    design_stems = np.asarray(stems)[design_idx]\n    design_labels = labels[design_idx]\n    design_groups = groups[design_idx]\n    fold_splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed + 1)\n    selected = list(fold_splitter.split(design_stems, design_labels, design_groups))[fold_index]\n    train_idx, val_idx = selected\n    return (\n        sorted(design_stems[train_idx].tolist()),\n        sorted(design_stems[val_idx].tolist()),\n        sorted(np.asarray(stems)[holdout_idx].tolist()),\n    )\n\n\ndef load_split_manifest(path, stems):\n    split_map = json.loads(Path(path).read_text())\n    if set(split_map) != {"train", "val", "test"}:\n        raise ValueError("Split manifest must contain train, val, and test")\n    flattened = [str(stem) for split in split_map.values() for stem in split]\n    if len(flattened) != len(set(flattened)):\n        raise ValueError("Split manifest contains duplicate IDs across splits")\n    if set(flattened) != set(stems):\n        raise ValueError("Split manifest IDs do not exactly match usable training IDs")\n    return {key: sorted(map(str, value)) for key, value in split_map.items()}\n\n\ndef assert_group_isolation(split_map, groups_by_stem):\n    group_splits = {}\n    for split, stems in split_map.items():\n        for stem in stems:\n            group = groups_by_stem[stem]\n            if group in group_splits and group_splits[group] != split:\n                raise ValueError(f"Group {group} leaks across {group_splits[group]} and {split}")\n            group_splits[group] = split\n\n\ndef write_label(path, rows):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    text = "\\n".join(\n        f"{int(c)} {cx:.8f} {cy:.8f} {w:.8f} {h:.8f}"\n        for c, cx, cy, w, h in rows\n    )\n    Path(path).write_text(text + ("\\n" if text else ""))\n\n\ndef image_id_value(stem):\n    return int(stem) if stem.isdigit() else stem\n\n\ndef parse_class_ids(value):\n    if not value:\n        return []\n    ids = sorted({int(item) for item in value.split(",")})\n    if any(class_id < 0 or class_id > 17 for class_id in ids):\n        raise argparse.ArgumentTypeError("Class IDs must be between 0 and 17")\n    return ids\n\n\ndef build_dataset(args):\n    work = Path(args.workdir)\n    work.mkdir(parents=True, exist_ok=True)\n    cache_root = Path(args.cache_dir) if args.cache_dir else work\n    prepared = cache_root / "dataset"\n    if prepared.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {prepared}")\n    names = read_classes(args.class_file)\n    class_to_id = {name: i for i, name in enumerate(names)}\n    train_cubes = collect_cubes(args.train_cubes)\n    design_only = getattr(args, "design_only", False)\n    test_cubes = [] if design_only else collect_cubes(args.test_cubes)\n    ranking_cubes = collect_cubes(args.ranking_cubes) if args.ranking_cubes and not design_only else []\n    xmls = annotation_map(args.train_xml)\n    cube_by_stem = unique_stems(train_cubes)\n    inference_by_stem = unique_stems(test_cubes + ranking_cubes)\n    if set(cube_by_stem) != set(xmls):\n        raise ValueError("Training image and XML IDs do not match")\n    stems = sorted(set(cube_by_stem) & set(xmls))\n    bad_stems = {"1227", "1836", "1855"}\n    stems = [s for s in stems if s not in bad_stems]\n    print(f"Skipping mismatched images: {sorted(bad_stems)}")\n\n    if not stems:\n        raise RuntimeError("No matching training cube/XML stems found")\n\n    boxes_by_stem = {}\n    dims_by_stem = {}\n    for stem in stems:\n        w, h, rows = parse_voc(xmls[stem], class_to_id)\n        boxes_by_stem[stem] = rows\n        dims_by_stem[stem] = (w, h)\n\n    groups_by_stem, groups_source = read_groups(args.groups_file, stems)\n    if args.split_manifest:\n        split_map = load_split_manifest(args.split_manifest, stems)\n    elif args.cv_fold_index is not None:\n        train_stems, val_stems, holdout_stems = cv_split(\n            stems,\n            boxes_by_stem,\n            groups_by_stem,\n            args.seed,\n            args.cv_folds,\n            args.cv_fold_index,\n        )\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    else:\n        train_stems, val_stems, holdout_stems = safe_split(stems, boxes_by_stem, args.seed)\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    assert_group_isolation(split_map, groups_by_stem)\n    train_stems = split_map["train"]\n    val_stems = split_map["val"]\n    holdout_stems = split_map["test"]\n    normalization = getattr(args, "normalization", "N0")\n    spectral_preprocess = getattr(args, "spectral_preprocess", "SP0")\n    pipeline_state = None\n    if normalization == "N0" and spectral_preprocess == "SP0":\n        lo, hi = band_limits([cube_by_stem[s] for s in train_stems], seed=args.seed)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import fit_pipeline, save_pipeline_state\n        except ModuleNotFoundError:\n            from pipeline import fit_pipeline, save_pipeline_state\n        pipeline_state = fit_pipeline(\n            (read_cube(cube_by_stem[s]) for s in train_stems),\n            normalization,\n            spectral_method=spectral_preprocess,\n            seed=args.seed,\n            samples_per_cube=getattr(args, "normalization_samples_per_image", 256),\n            mnf_components=getattr(args, "mnf_components", None),\n            savgol_window=getattr(args, "savgol_window", 5),\n            savgol_order=getattr(args, "savgol_order", 2),\n        )\n        lo = hi = None\n        save_pipeline_state(pipeline_state, work / "spectral_pipeline_state.json")\n    input_policy = resolve_input_policy(args.family, args.input_policy)\n    (work / "split_manifest.json").write_text(json.dumps(split_map, indent=2))\n    train_images_written = 0\n    for split, split_stems in split_map.items():\n        if design_only and split == "test":\n            continue\n        for stem in split_stems:\n            cube = read_cube(cube_by_stem[stem])\n            h, w = cube.shape[:2]\n            ann_w, ann_h = dims_by_stem[stem]\n            if (w, h) != (ann_w, ann_h):\n                raise ValueError(\n                    f"Spatial mismatch for {stem}: cube {(w, h)} vs annotation {(ann_w, ann_h)}"\n                )\n            geometry = write_tiff(\n                cube,\n                prepared / "images" / split / f"{stem}.tiff",\n                lo,\n                hi,\n                input_policy,\n                pipeline_state,\n            )\n            rows = transform_rows(boxes_by_stem[stem], geometry)\n            write_label(prepared / "labels" / split / f"{stem}.txt", rows)\n            if split != "train":\n                continue\n            train_images_written += 1\n            classes = {int(row[0]) for row in boxes_by_stem[stem]}\n            selected = not args.spectral_aug_classes or classes.intersection(args.spectral_aug_classes)\n            if not selected:\n                continue\n            for copy_index in range(args.spectral_aug_copies):\n                suffix = f"{stem}_hsaug{copy_index + 1}"\n                augmented = spectral_augment(\n                    cube,\n                    stable_rng(args.seed, stem, copy_index),\n                    args.spectral_gain,\n                    args.spectral_tilt,\n                    args.spectral_noise,\n                )\n                write_tiff(\n                    augmented,\n                    prepared / "images" / split / f"{suffix}.tiff",\n                    lo,\n                    hi,\n                    input_policy,\n                    pipeline_state,\n                )\n                write_label(prepared / "labels" / split / f"{suffix}.txt", rows)\n                train_images_written += 1\n\n    test_out = cache_root / "test_tiff"\n    if test_out.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {test_out}")\n    test_out.mkdir(parents=True, exist_ok=True)\n    inference_geometry = {}\n    for stem, path in inference_by_stem.items():\n        inference_geometry[stem] = write_tiff(\n            read_cube(path),\n            test_out / f"{stem}.tiff",\n            lo,\n            hi,\n            input_policy,\n            pipeline_state,\n        )\n    (work / "inference_geometry.json").write_text(json.dumps(inference_geometry, indent=2))\n\n    data = {\n        "path": str(prepared),\n        "train": "images/train",\n        "val": "images/val",\n        "test": "images/test",\n        "channels": 16,\n        "names": {i: n for i, n in enumerate(names)},\n    }\n    yaml_path = work / "dataset.yaml"\n    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))\n\n    report = {\n        "train_images_total": len(stems),\n        "train_split": len(train_stems),\n        "val_split": len(val_stems),\n        "holdout_split": len(holdout_stems),\n        "prepared_train_images": train_images_written,\n        "competition_test_images": len(test_cubes),\n        "competition_ranking_images": len(ranking_cubes),\n        "class_names": names,\n        "excluded_ids": sorted(bad_stems),\n        "split_ids": split_map,\n        "split_mode": "manifest" if args.split_manifest else ("cv" if args.cv_fold_index is not None else "legacy"),\n        "cv_folds": args.cv_folds if args.cv_fold_index is not None else None,\n        "cv_fold_index": args.cv_fold_index,\n        "groups_source": groups_source,\n        "group_count": len(set(groups_by_stem.values())),\n        "input_policy": input_policy,\n        "spectral_augmentation": {\n            "copies": args.spectral_aug_copies,\n            "classes": args.spectral_aug_classes,\n            "gain_limit": args.spectral_gain,\n            "tilt_limit": args.spectral_tilt,\n            "noise_std": args.spectral_noise,\n        },\n        "class_counts": {split: np.bincount([int(row[0]) for stem in ids for row in boxes_by_stem[stem]], minlength=18).tolist() for split, ids in split_map.items()},\n        "band_low_0_5pct": lo.tolist() if lo is not None else None,\n        "band_high_99_5pct": hi.tolist() if hi is not None else None,\n        "normalization": normalization,\n        "spectral_preprocess": spectral_preprocess,\n        "spectral_pipeline_state": pipeline_state.to_dict() if pipeline_state is not None else None,\n    }\n    (work / "data_audit.json").write_text(json.dumps(report, indent=2))\n    return yaml_path, test_out, inference_geometry\n\n\ndef train(args, yaml_path):\n    model = model_class(args)(args.model)\n    result = model.train(\n        trainer=trainer_class(args),\n        data=str(yaml_path),\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=args.run_name,\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=args.multi_scale,\n        patience=25,\n        close_mosaic=15,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=5.0,\n        translate=0.08,\n        scale=0.35,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.5,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef localization_finetune(args, yaml_path, checkpoint):\n    if args.localization_epochs <= 0:\n        return None\n    model = model_class(args)(str(checkpoint))\n    result = model.train(\n        data=str(yaml_path),\n        epochs=args.localization_epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=f"{args.run_name}_localize",\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=0.0,\n        patience=args.localization_epochs,\n        lr0=args.localization_lr0,\n        optimizer="AdamW",\n        lrf=0.1,\n        cos_lr=True,\n        warmup_epochs=0.5,\n        close_mosaic=0,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=0.0,\n        translate=0.02,\n        scale=0.10,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.0,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef predict(args, best, test_out, inference_geometry, tta):\n    model = model_class(args)(str(best))\n    results = model.predict(\n        source=str(test_out),\n        imgsz=args.imgsz,\n        conf=args.predict_conf,\n        iou=args.iou,\n        max_det=args.predict_max_det,\n        device=args.device,\n        augment=tta,\n        stream=True,\n        verbose=False,\n    )\n    rows = []\n    seen = set()\n    empty = []\n    for r in results:\n        seen.add(Path(r.path).stem)\n        if r.boxes is None or len(r.boxes) == 0:\n            empty.append(Path(r.path).stem)\n        stem = Path(r.path).stem\n        if r.boxes is None:\n            continue\n        xyxy = restore_boxes(\n            r.boxes.xyxy.detach().cpu().numpy(),\n            inference_geometry[stem],\n        )\n        confs = r.boxes.conf.detach().cpu().numpy()\n        classes = r.boxes.cls.detach().cpu().numpy().astype(int)\n        for box, score, cls in zip(xyxy, confs, classes):\n            x1, y1, x2, y2 = map(float, box)\n            if x2 <= x1 or y2 <= y1:\n                continue\n            rows.append(\n                {\n                    "image_id": image_id_value(stem),\n                    "class_id": int(cls),\n                    "confidence": float(score),\n                    "x1": x1,\n                    "y1": y1,\n                    "x2": x2,\n                    "y2": y2,\n                }\n            )\n\n    expected_ids = set(inference_geometry)\n    if seen != expected_ids:\n        raise RuntimeError("Some test images were not processed")\n    counts = pd.Series([row["image_id"] for row in rows]).value_counts()\n    (Path(args.workdir) / "prediction_audit.json").write_text(json.dumps({\n        "processed_images": len(seen),\n        "no_detection_ids": empty,\n        "prediction_confidence": args.predict_conf,\n        "prediction_max_det": args.predict_max_det,\n        "tta": tta,\n        "predictions_total": len(rows),\n        "predictions_per_image_min": int(counts.min()) if len(counts) else 0,\n        "predictions_per_image_median": float(counts.median()) if len(counts) else 0.0,\n        "predictions_per_image_max": int(counts.max()) if len(counts) else 0,\n    }, indent=2))\n    sub = pd.DataFrame(rows)\n    if sub.empty:\n        raise RuntimeError("No detections produced")\n    sub.insert(0, "id", np.arange(len(sub), dtype=int))\n    expected = ["id", "image_id", "class_id", "confidence", "x1", "y1", "x2", "y2"]\n    sub = sub[expected]\n    if args.sample_submission and pd.read_csv(args.sample_submission, nrows=0).columns.tolist() != expected:\n        raise ValueError("Sample submission columns do not match")\n    if not np.isfinite(sub[["confidence", "x1", "y1", "x2", "y2"]].to_numpy()).all():\n        raise ValueError("Non-finite submission values")\n    if not sub["id"].is_unique:\n        raise RuntimeError("Submission id is not unique")\n    if not sub["class_id"].between(0, 17).all():\n        raise RuntimeError("class_id outside 0..17")\n    if not sub["confidence"].between(0, 1).all():\n        raise RuntimeError("confidence outside 0..1")\n    if not ((sub["x2"] > sub["x1"]) & (sub["y2"] > sub["y1"])).all():\n        raise RuntimeError("Invalid boxes")\n    out_path = Path(args.workdir) / "submission.csv"\n    sub.to_csv(out_path, index=False)\n    return out_path\n\n\ndef unique_stems(files):\n    result = {}\n    for path in files:\n        if path.stem in result:\n            raise ValueError(f"Duplicate image or XML ID: {path.stem}")\n        result[path.stem] = path\n    return result\n\n\ndef model_class(args):\n    from ultralytics import RTDETR, YOLO\n\n    return RTDETR if args.family == "rtdetr" else YOLO\n\n\ndef metric_summary(metrics):\n    class_ids = [int(c) for c in metrics.box.ap_class_index]\n    per_class = {}\n    for position, class_id in enumerate(class_ids):\n        per_class[str(class_id)] = {\n            "ap50_95": float(metrics.box.ap[position]),\n            "ap50": float(metrics.box.ap50[position]),\n            "ap75": float(metrics.box.all_ap[position, 5]),\n            "precision": float(metrics.box.p[position]),\n            "recall": float(metrics.box.r[position]),\n        }\n    summary = {\n        "map50_95": float(metrics.box.map),\n        "map50": float(metrics.box.map50),\n        "map75": float(metrics.box.map75),\n        "localization_gap_map50_minus_map75": float(metrics.box.map50 - metrics.box.map75),\n        "per_class": per_class,\n        "worst_present_class_ap": min((row["ap50_95"] for row in per_class.values()), default=None),\n        "absent_class_ids": sorted(set(range(18)) - set(class_ids)),\n    }\n    matrix = np.asarray(metrics.confusion_matrix.matrix)\n    if matrix.shape == (19, 19) and matrix.sum() > 0:\n        true_positive = np.diag(matrix)[:18]\n        summary["false_positives_per_class"] = (\n            matrix[:18, :].sum(axis=1) - true_positive\n        ).astype(int).tolist()\n        summary["false_negatives_per_class"] = (\n            matrix[:, :18].sum(axis=0) - true_positive\n        ).astype(int).tolist()\n        summary["confusion_matrix_predicted_rows_true_columns"] = matrix.astype(int).tolist()\n    else:\n        summary["false_positives_per_class"] = None\n        summary["false_negatives_per_class"] = None\n        summary["confusion_matrix_predicted_rows_true_columns"] = None\n        summary["confusion_status"] = "unavailable; compute from saved predictions at an explicit operating point"\n    values = [summary["map50_95"], summary["map50"], summary["map75"]]\n    if not np.isfinite(values).all():\n        raise RuntimeError("Non-finite evaluation metrics")\n    return summary\n\n\ndef evaluate(args, best, yaml_path, splits, output_name, tta=False):\n    model = model_class(args)(str(best))\n    report = {\n        "checkpoint": str(best),\n        "sha256": hashlib.sha256(Path(best).read_bytes()).hexdigest(),\n        "tta": tta,\n    }\n    for split in splits:\n        metrics = model.val(data=str(yaml_path), split=split, imgsz=args.imgsz,\n                            batch=args.batch if args.batch > 0 else 4,\n                            device=args.device, workers=args.workers, conf=args.conf,\n                            max_det=args.max_det, iou=args.iou, plots=False,\n                            augment=tta,\n                            project=str(Path(args.workdir) / "evaluation"),\n                            name=f"{Path(output_name).stem}_{split}")\n        report[split] = metric_summary(metrics)\n        (Path(args.workdir) / output_name).write_text(json.dumps(report, indent=2))\n    return report\n\n\ndef promote_candidate(base_report, candidate_report, min_gain):\n    base = base_report["val"]\n    candidate = candidate_report["val"]\n    gain = candidate["map50_95"] - base["map50_95"]\n    map75_gain = candidate["map75"] - base["map75"]\n    worst_regression = candidate["worst_present_class_ap"] - base["worst_present_class_ap"]\n    passed = (gain >= min_gain or (gain >= 0 and map75_gain >= min_gain)) and worst_regression >= -0.03\n    return {\n        "passed": passed,\n        "map50_95_gain": gain,\n        "map75_gain": map75_gain,\n        "worst_present_class_ap_change": worst_regression,\n        "minimum_gain": min_gain,\n    }\n\n\ndef check_submission_gate(args, report):\n    holdout_map = report["test"]["map50_95"]\n    gate = {\n        "metric": "holdout_mAP50_95",\n        "value": holdout_map,\n        "minimum": args.min_holdout_map,\n        "passed": holdout_map >= args.min_holdout_map,\n    }\n    (Path(args.workdir) / "submission_gate.json").write_text(json.dumps(gate, indent=2))\n    return gate\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--train-cubes", required=True)\n    p.add_argument("--test-cubes", required=True)\n    p.add_argument("--ranking-cubes")\n    p.add_argument("--train-xml", required=True)\n    p.add_argument("--class-file", required=True)\n    p.add_argument("--workdir", default="/kaggle/working/hsi_submission01")\n    p.add_argument("--cache-dir")\n    p.add_argument("--groups-file")\n    p.add_argument("--split-manifest")\n    p.add_argument("--cv-folds", type=int, default=3)\n    p.add_argument("--cv-fold-index", type=int)\n    p.add_argument("--model", default="yolo26m.pt")\n    p.add_argument("--checkpoint")\n    p.add_argument("--epochs", type=int, default=120)\n    p.add_argument("--imgsz", type=int, default=1024)\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--device", default="0")\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--input-policy", choices=["auto", "native", "square_pad"], default="auto")\n    p.add_argument("--stem-init", choices=["random", "rgb_mean"], default="rgb_mean")\n    p.add_argument("--normalization", choices=["N0", "N1", "N2", "N3", "N4"], default="N0")\n    p.add_argument("--spectral-preprocess", choices=["SP0", "SP1", "SP2", "SP3"], default="SP0")\n    p.add_argument("--normalization-samples-per-image", type=int, default=256)\n    p.add_argument("--mnf-components", type=int)\n    p.add_argument("--savgol-window", type=int, default=5)\n    p.add_argument("--savgol-order", type=int, default=2)\n    p.add_argument("--multi-scale", type=float, default=0.0)\n    p.add_argument("--spectral-aug-copies", type=int, default=0)\n    p.add_argument("--spectral-aug-classes", type=parse_class_ids, default=[])\n    p.add_argument("--spectral-gain", type=float, default=0.05)\n    p.add_argument("--spectral-tilt", type=float, default=0.03)\n    p.add_argument("--spectral-noise", type=float, default=0.01)\n    p.add_argument("--localization-epochs", type=int, default=0)\n    p.add_argument("--localization-lr0", type=float, default=0.0001)\n    p.add_argument("--promotion-min-gain", type=float, default=0.005)\n    p.add_argument("--tta", action="store_true")\n    p.add_argument("--tta-min-gain", type=float, default=0.002)\n    p.add_argument("--design-only", action="store_true")\n    p.add_argument("--conf", type=float, default=0.001)\n    p.add_argument("--predict-conf", type=float, default=0.001)\n    p.add_argument("--iou", type=float, default=0.65)\n    p.add_argument("--max-det", type=int, default=300)\n    p.add_argument("--predict-max-det", type=int, default=300)\n    p.add_argument("--min-holdout-map", type=float, default=0.50)\n    p.add_argument("--family", choices=["yolo", "rtdetr"], default="yolo")\n    p.add_argument("--batch", type=int, default=-1)\n    p.add_argument("--run-name", default="submission01")\n    p.add_argument("--sample-submission")\n    p.add_argument("--evaluate-holdout", action="store_true")\n    args = p.parse_args()\n    if args.spectral_aug_copies < 0:\n        raise ValueError("spectral_aug_copies must be non-negative")\n    if args.spectral_preprocess == "SP1" and args.mnf_components is None:\n        raise ValueError("SP1 requires --mnf-components selected on validation")\n    if not 0 <= args.multi_scale <= 0.9:\n        raise ValueError("multi_scale must be in 0..0.9")\n    if args.tta and args.family != "yolo":\n        raise ValueError("Built-in validated TTA is enabled only for YOLO in this runner")\n    import torch\n    if args.device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("GPU is unavailable. Enable a Kaggle GPU and keep its CUDA PyTorch installation.")\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    Path(args.workdir).mkdir(parents=True, exist_ok=True)\n\n    versions = {name: importlib.metadata.version(name) for name in ("torch", "ultralytics", "numpy")}\n    run_config = {\n        "args": vars(args),\n        "resolved_input_policy": resolve_input_policy(args.family, args.input_policy),\n        "versions": versions,\n    }\n    (Path(args.workdir) / "run_config.json").write_text(json.dumps(run_config, indent=2))\n    yaml_path, test_out, inference_geometry = build_dataset(args)\n    if args.checkpoint:\n        base_checkpoint = Path(args.checkpoint)\n        if not base_checkpoint.exists():\n            raise FileNotFoundError(base_checkpoint)\n    else:\n        base_checkpoint = train(args, yaml_path)\n    base_report = evaluate(\n        args,\n        base_checkpoint,\n        yaml_path,\n        ["val"],\n        "evaluation_base.json",\n    )\n    selected_checkpoint = base_checkpoint\n    selected_report = base_report\n    selection = {"base_checkpoint": str(base_checkpoint), "localization_finetune": None, "tta": None}\n    finetuned_checkpoint = localization_finetune(args, yaml_path, base_checkpoint)\n    if finetuned_checkpoint:\n        finetuned_report = evaluate(\n            args,\n            finetuned_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_localized.json",\n        )\n        decision = promote_candidate(base_report, finetuned_report, args.promotion_min_gain)\n        decision["checkpoint"] = str(finetuned_checkpoint)\n        selection["localization_finetune"] = decision\n        if decision["passed"]:\n            selected_checkpoint = finetuned_checkpoint\n            selected_report = finetuned_report\n    selected_tta = False\n    if args.tta:\n        tta_report = evaluate(\n            args,\n            selected_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_tta.json",\n            tta=True,\n        )\n        decision = promote_candidate(selected_report, tta_report, args.tta_min_gain)\n        selection["tta"] = decision\n        if decision["passed"]:\n            selected_report = tta_report\n            selected_tta = True\n    selection["selected_checkpoint"] = str(selected_checkpoint)\n    selection["selected_checkpoint_sha256"] = hashlib.sha256(Path(selected_checkpoint).read_bytes()).hexdigest()\n    selection["selected_tta"] = selected_tta\n    (Path(args.workdir) / "selection.json").write_text(json.dumps(selection, indent=2))\n    if args.design_only:\n        print(selection)\n        print("Design-only run complete. Holdout and competition test prediction were not used.")\n        return\n    holdout_report = evaluate(\n        args,\n        selected_checkpoint,\n        yaml_path,\n        ["test"],\n        "evaluation_holdout.json",\n        tta=selected_tta,\n    )\n    report = {\n        "checkpoint": str(selected_checkpoint),\n        "sha256": selection["selected_checkpoint_sha256"],\n        "tta": selected_tta,\n        "val": selected_report["val"],\n        "test": holdout_report["test"],\n    }\n    (Path(args.workdir) / "evaluation.json").write_text(json.dumps(report, indent=2))\n    gate = check_submission_gate(args, report)\n    print(gate)\n    if not gate["passed"]:\n        print("Submission blocked: holdout mAP50-95 is below minimum.")\n        print(selected_checkpoint)\n        return\n    submission = predict(args, selected_checkpoint, test_out, inference_geometry, selected_tta)\n    print(selected_checkpoint)\n    print(submission)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/build_folds.py').write_text('import argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\n\n\nEXCLUDED_IDS = ("1227", "1836", "1855")\nAREA_EDGES = (0, 32**2, 96**2)\n\n\ndef digest_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef unique_files(root, suffixes):\n    result = {}\n    for path in sorted(Path(root).rglob("*")):\n        if path.is_file() and path.suffix.lower() in suffixes:\n            if path.stem in result:\n                raise ValueError(f"Duplicate image ID {path.stem}: {result[path.stem]}, {path}")\n            result[path.stem] = path\n    if not result:\n        raise ValueError(f"No matching files in {root}")\n    return result\n\n\ndef non_object_metadata(root):\n    result = defaultdict(list)\n\n    def visit(node, prefix):\n        if node.tag.lower() == "object":\n            return\n        key = f"{prefix}/{node.tag}" if prefix else node.tag\n        result[key].append({"text": (node.text or "").strip(), "attributes": dict(node.attrib)})\n        for child in node:\n            visit(child, key)\n\n    visit(root, "")\n    return dict(result)\n\n\ndef read_annotation(path, names):\n    root = ET.parse(path).getroot()\n    width = float(root.findtext("size/width"))\n    height = float(root.findtext("size/height"))\n    if not np.isfinite([width, height]).all() or min(width, height) <= 0:\n        raise ValueError(f"Invalid original dimensions in {path}")\n    counts = np.zeros(len(names), dtype=np.int64)\n    areas = []\n    class_sizes = np.zeros((len(names), 3), dtype=np.int64)\n    clipped = discarded = 0\n    for obj in root.findall("object"):\n        name = (obj.findtext("name") or "").strip()\n        if name not in names:\n            raise ValueError(f"Unknown class {name!r} in {path}")\n        box = np.array([float(obj.findtext(f"bndbox/{key}")) for key in ("xmin", "ymin", "xmax", "ymax")])\n        if not np.isfinite(box).all():\n            raise ValueError(f"Non-finite box in {path}")\n        bounded = np.clip(box, 0, [width, height, width, height])\n        clipped += int(not np.array_equal(box, bounded))\n        x1, y1, x2, y2 = bounded\n        if x2 <= x1 or y2 <= y1:\n            discarded += 1\n            continue\n        area = float((x2 - x1) * (y2 - y1))\n        class_id = names.index(name)\n        counts[class_id] += 1\n        areas.append(area)\n        class_sizes[class_id, int(np.searchsorted(AREA_EDGES[1:], area, side="right"))] += 1\n    return {\n        "width": width, "height": height, "class_counts": counts.tolist(),\n        "class_presence": (counts > 0).astype(int).tolist(), "box_areas": areas,\n        "class_size_counts": class_sizes.tolist(), "size_counts": class_sizes.sum(0).tolist(),\n        "clipped_box_count": clipped, "discarded_box_count": discarded,\n        "metadata": non_object_metadata(root),\n    }\n\n\ndef load_groups(ids, groups_file=None, groups_evidence=None):\n    if groups_file is None:\n        if groups_evidence is not None:\n            raise ValueError("--groups-evidence requires --groups-file")\n        return {stem: stem for stem in ids}, {\n            "mode": "image_id_only",\n            "relationship_origin": "identity_fallback",\n            "source_grouping_verified": False,\n            "verification_status": "no_source_relationship_claimed",\n        }\n    if groups_evidence is None or not Path(groups_evidence).read_text().strip():\n        raise ValueError("A group mapping requires a non-empty --groups-evidence file documenting the relationship")\n    with Path(groups_file).open(newline="") as handle:\n        reader = csv.DictReader(handle)\n        if set(reader.fieldnames or []) != {"image_id", "group_id"}:\n            raise ValueError("Groups CSV must contain exactly image_id,group_id")\n        groups = {}\n        for row in reader:\n            stem, group = row["image_id"].strip(), row["group_id"].strip()\n            if not stem or not group or stem in groups:\n                raise ValueError("Empty or duplicate image ID/group in groups CSV")\n            groups[stem] = group\n    if set(ids) - groups.keys():\n        raise ValueError(f"Groups CSV missing IDs: {sorted(set(ids) - groups.keys())[:10]}")\n    return {stem: groups[stem] for stem in ids}, {\n        "mode": "provided_group_mapping",\n        "relationship_origin": "caller_provided_csv",\n        "source_grouping_verified": False,\n        "verification_status": "requires_evidence_review",\n        "groups_file_sha256": digest_file(groups_file),\n        "groups_evidence_sha256": digest_file(groups_evidence),\n        "groups_evidence": Path(groups_evidence).read_text(),\n        "unused_group_ids": sorted(set(groups) - set(ids)),\n    }\n\n\ndef check_split(split, ids, groups):\n    flat = [stem for part in ("train", "val", "test") for stem in split[part]]\n    if len(flat) != len(set(flat)) or set(flat) != set(ids):\n        raise ValueError("Split must cover every usable image exactly once")\n    seen = {}\n    for part in ("train", "val", "test"):\n        if not split[part]:\n            raise ValueError(f"Empty {part} partition")\n        for stem in split[part]:\n            group = groups[stem]\n            if group in seen and seen[group] != part:\n                raise ValueError(f"Group {group} leaks across partitions")\n            seen[group] = part\n\n\ndef assign_groups(ids, groups, records, seed):\n    members = defaultdict(list)\n    for stem in sorted(ids):\n        members[groups[stem]].append(stem)\n    if len(members) < 3:\n        raise ValueError("Three-fold CV requires at least three development groups")\n    keys = sorted(members)\n    vectors = np.asarray([\n        np.sum([records[s]["class_presence"] + records[s]["class_counts"] + records[s]["size_counts"] + [1] for s in members[g]], axis=0).tolist() + [1]\n        for g in keys\n    ], dtype=float)\n    total = vectors.sum(0)\n    active = total > 0\n    normalized = vectors[:, active] / total[active]\n    rng = np.random.default_rng(seed)\n    ties = rng.random(len(keys))\n    order = sorted(range(len(keys)), key=lambda i: (-float(np.max(normalized[i])), -float(np.sum(normalized[i])), ties[i]))\n    loads = np.zeros((3, normalized.shape[1]))\n    counts = np.zeros(3, dtype=int)\n    chosen = {}\n    priority = rng.permutation(3).tolist()\n    for i in order:\n        scores = []\n        for fold in range(3):\n            trial = loads.copy()\n            trial[fold] += normalized[i]\n            scores.append(float(np.square(trial - trial.mean(0)).sum()))\n        fold = min(range(3), key=lambda f: (scores[f], counts[f], priority.index(f)))\n        loads[fold] += normalized[i]\n        counts[fold] += len(members[keys[i]])\n        chosen[keys[i]] = fold\n    return {stem: chosen[groups[stem]] for stem in ids}\n\n\ndef summarize(ids, records, groups, names):\n    counts = np.sum([records[s]["class_counts"] for s in ids], axis=0).astype(int)\n    presence = np.sum([records[s]["class_presence"] for s in ids], axis=0).astype(int)\n    areas = [area for s in ids for area in records[s]["box_areas"]]\n    histogram, _ = np.histogram(areas, bins=[0, 256, 1024, 4096, 9216, 16384, 65536, np.inf])\n    return {\n        "image_count": len(ids), "group_count": len({groups[s] for s in ids}),\n        "object_count": int(counts.sum()), "class_object_counts": dict(zip(names, counts.tolist())),\n        "class_image_counts": dict(zip(names, presence.tolist())),\n        "missing_classes": [name for name, count in zip(names, counts) if count == 0],\n        "size_counts": dict(zip(("small", "medium", "large"), np.sum([records[s]["size_counts"] for s in ids], axis=0).astype(int).tolist())),\n        "class_size_counts": dict(zip(names, np.sum([records[s]["class_size_counts"] for s in ids], axis=0).astype(int).tolist())),\n        "box_area_histogram": {"edges": [0, 256, 1024, 4096, 9216, 16384, 65536, "infinity"], "counts": histogram.tolist()},\n    }\n\n\ndef build_folds(xml_dir, cube_dir, class_file, legacy_split, output_dir, groups_file=None, groups_evidence=None, seed=42):\n    names = [line.strip() for line in Path(class_file).read_text().splitlines() if line.strip()]\n    if not names or len(names) != len(set(names)):\n        raise ValueError("Class names must be nonempty and unique")\n    xmls = unique_files(xml_dir, {".xml"})\n    cubes = unique_files(cube_dir, {".png", ".npy", ".npz", ".tif", ".tiff"})\n    if set(xmls) != set(cubes):\n        raise ValueError("Annotation and cube image IDs differ")\n    ids = sorted(set(xmls) - set(EXCLUDED_IDS))\n    legacy = json.loads(Path(legacy_split).read_text())\n    if set(legacy) != {"train", "val", "test"}:\n        raise ValueError("Legacy split requires train, val, test")\n    legacy = {key: list(map(str, value)) for key, value in legacy.items()}\n    check_split(legacy, ids, {stem: stem for stem in ids})\n    records = {stem: read_annotation(xmls[stem], names) for stem in ids}\n    groups, group_info = load_groups(ids, groups_file, groups_evidence)\n    holdout_groups = {groups[stem] for stem in legacy["test"]}\n    holdout = sorted(stem for stem in ids if groups[stem] in holdout_groups)\n    development = sorted(set(ids) - set(holdout))\n    assignments = assign_groups(development, groups, records, seed)\n    folds = []\n    for index in range(3):\n        split = {"train": [s for s in development if assignments[s] != index], "val": [s for s in development if assignments[s] == index], "test": holdout}\n        check_split(split, ids, groups)\n        folds.append(split)\n    if sorted(s for split in folds for s in split["val"]) != development:\n        raise AssertionError("Development images must validate exactly once")\n    fields = defaultdict(Counter)\n    inventory = {}\n    for stem, record in records.items():\n        for key, values in record["metadata"].items():\n            fields[key].update(json.dumps(v, sort_keys=True) for v in values)\n        inventory[stem] = {\n            "xml": str(xmls[stem].relative_to(xml_dir)), "cube": str(cubes[stem].relative_to(cube_dir)),\n            "xml_sha256": digest_file(xmls[stem]), "cube_size_bytes": cubes[stem].stat().st_size,\n            "xml_metadata": record["metadata"], "original_width": record["width"], "original_height": record["height"],\n            "original_box_areas": record["box_areas"], "clipped_box_count": record["clipped_box_count"],\n            "discarded_box_count": record["discarded_box_count"],\n        }\n    grouping_fields = [key for key in fields if any(word in key.lower() for word in ("capture", "session", "source", "folder", "sequence", "scene", "video"))]\n    candidate_fields = [key for key in grouping_fields if len(fields[key]) > 1]\n    non_informative_fields = [key for key in grouping_fields if len(fields[key]) <= 1]\n    config = {"seed": seed, "cv_folds": 3, "excluded_ids": list(EXCLUDED_IDS), "area_bins": list(AREA_EDGES),\n              "legacy_split_sha256": digest_file(legacy_split), "class_file_sha256": digest_file(class_file),\n              "group_mapping_sha256": digest_json(groups), "annotation_inventory_sha256": digest_json(inventory),\n              "groups_evidence_sha256": group_info.get("groups_evidence_sha256"),\n              "algorithm": "seeded_greedy_multilabel_presence_object_counts_original_area_bins_image_and_group_counts"}\n    audit = {\n        "status": "folds_built_not_training_results", "code_hash": digest_file(__file__), "config": config,\n        "config_hash": digest_json(config), "groups": group_info,\n        "excluded_ids": {stem: {"present": stem in xmls,\n                                "reason": "Existing runner exclusion retained for comparability; the historical mismatch rationale was not re-verified",\n                                "evidence_status": "historical_policy_unverified"} for stem in EXCLUDED_IDS},\n        "holdout": {"status": "previously_inspected", "legacy_count": len(legacy["test"]), "group_closed_count": len(holdout),\n                    "added_for_group_closure": sorted(set(holdout) - set(legacy["test"])),\n                    "previously_trained_by_legacy_checkpoint": sorted(set(holdout) & set(legacy["train"])),\n                    "legacy_checkpoint_declared_group_overlap": sorted(holdout_groups & {groups[s] for s in legacy["train"]}),\n                    "use": "Freeze new designs before evaluation; cannot claim an untouched final holdout. Existing checkpoints cannot provide new CV evidence."},\n        "metadata_audit": {"non_object_fields": {key: {"unique_values": len(values), "value_counts": dict(sorted(values.items()))} for key, values in sorted(fields.items())},\n                           "candidate_group_fields": candidate_fields,\n                           "non_informative_group_fields": non_informative_fields,\n                           "unresolved_metadata_candidates": candidate_fields if groups_file is None else [],\n                           "xml_folders": dict(Counter(str(path.parent.relative_to(xml_dir)) for path in xmls.values())),\n                           "cube_folders": dict(Counter(str(path.parent.relative_to(cube_dir)) for path in cubes.values())),\n                           "numeric_filename_count": sum(stem.isdigit() for stem in ids),\n                           "numeric_name_policy": "Numbers alone are not capture/session evidence; no numeric or visual grouping inferred",\n                           "conclusion": ("No varying official capture/session/source field exists; image_id_only is the supported fallback" if not candidate_fields else "Only image identities are currently supported; varying source/session candidates require provenance review") if groups_file is None else "Caller-provided mapping used for isolation; no source relationship is claimed until attached provenance is reviewed"},\n        "coordinate_policy": "Original XML pixel coordinates; clip to XML image bounds; discard nonpositive boxes as existing runner does; no resize before areas",\n        "assertions": {"image_disjoint": True, "declared_group_mapping_disjoint": True,\n                       "verified_source_group_disjoint": True if group_info["source_grouping_verified"] else None,\n                       "all_usable_ids_covered": True, "validation_once_per_development_id": True,\n                       "legacy_holdout_preserved": True},\n        "folds": [{"fold": i, "split_hash": digest_json(split), "partitions": {part: summarize(stems, records, groups, names) for part, stems in split.items()}} for i, split in enumerate(folds)],\n    }\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    paths = [out / f"fold{i}.json" for i in range(3)] + [out / "groups.csv", out / "fold_audit.json", out / "metadata_inventory.json"]\n    if any(path.exists() for path in paths):\n        raise FileExistsError("Use a fresh output directory; frozen fold artifacts cannot be overwritten")\n    for i, split in enumerate(folds):\n        (out / f"fold{i}.json").write_text(json.dumps(split, indent=2) + "\\n")\n    with (out / "groups.csv").open("w", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["image_id", "group_id"])\n        writer.writerows((stem, groups[stem]) for stem in ids)\n    (out / "fold_audit.json").write_text(json.dumps(audit, indent=2, allow_nan=False) + "\\n")\n    (out / "metadata_inventory.json").write_text(json.dumps(inventory, indent=2, allow_nan=False) + "\\n")\n    return audit\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--xml-dir", required=True)\n    parser.add_argument("--cube-dir", required=True)\n    parser.add_argument("--class-file", required=True)\n    parser.add_argument("--legacy-split", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--groups-file")\n    parser.add_argument("--groups-evidence")\n    parser.add_argument("--seed", type=int, default=42)\n    audit = build_folds(**vars(parser.parse_args()))\n    print(json.dumps({"status": audit["status"], "grouping": audit["groups"]["mode"], "fold_sizes": [{part: value["image_count"] for part, value in fold["partitions"].items()} for fold in audit["folds"]], "holdout": audit["holdout"]["status"]}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/architecture_dataset.py').write_text('import argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef build_dataset(args):\n    from hsi_runner import (\n        annotation_map,\n        band_limits,\n        collect_cubes,\n        parse_voc,\n        prepare_cube,\n        read_classes,\n        read_cube,\n        transform_rows,\n        unique_stems,\n    )\n\n    cube_by_stem = unique_stems(collect_cubes(args.cube_dir))\n    xml_by_stem = annotation_map(args.xml_dir)\n    names = read_classes(args.class_file)\n    class_to_id = {name: index for index, name in enumerate(names)}\n    split_map = json.loads(Path(args.split_manifest).read_text())\n    if set(split_map) != {"train", "val", "test"}:\n        raise ValueError("Split manifest must contain train, val, and test")\n    selected = sorted(set(split_map["train"]) | set(split_map["val"]))\n    missing = sorted(set(selected) - set(cube_by_stem))\n    if missing:\n        raise ValueError(f"Missing {len(missing)} cubes from split manifest")\n    missing_xml = sorted(set(selected) - set(xml_by_stem))\n    if missing_xml:\n        raise ValueError(f"Missing {len(missing_xml)} XML files from split manifest")\n\n    train_files = [cube_by_stem[str(stem)] for stem in split_map["train"]]\n    lo, hi = band_limits(train_files, seed=args.seed)\n    output = Path(args.output_dir)\n    annotations_dir = output / "annotations"\n    annotations_dir.mkdir(parents=True, exist_ok=True)\n    categories = [{"id": index, "name": name} for index, name in enumerate(names)]\n    evidence = {\n        "split_manifest": str(Path(args.split_manifest)),\n        "split_sha256": sha256(args.split_manifest),\n        "class_file_sha256": sha256(args.class_file),\n        "input_policy": "square_pad",\n        "normalization": "train-only percentile 0.5/99.5 to uint8",\n        "train_images": len(split_map["train"]),\n        "val_images": len(split_map["val"]),\n        "holdout_images_not_used": len(split_map["test"]),\n        "band_lo": lo.tolist(),\n        "band_hi": hi.tolist(),\n    }\n\n    for split in ("train", "val"):\n        image_dir = output / "images" / split\n        image_dir.mkdir(parents=True, exist_ok=True)\n        images = []\n        annotations = []\n        for image_index, stem in enumerate(map(str, split_map[split])):\n            cube = read_cube(cube_by_stem[stem])\n            xml_width, xml_height, rows = parse_voc(xml_by_stem[stem], class_to_id)\n            if cube.shape[:2] != (xml_height, xml_width):\n                raise ValueError(\n                    f"{stem}: cube {cube.shape[:2]} differs from XML {(xml_height, xml_width)}"\n                )\n            prepared, geometry = prepare_cube(cube, lo, hi, "square_pad")\n            prepared = np.ascontiguousarray(np.moveaxis(prepared, -1, 0))\n            file_name = f"{stem}.npy"\n            np.save(image_dir / file_name, prepared, allow_pickle=False)\n            image_id = image_index + 1\n            height, width = prepared.shape[-2:]\n            images.append(\n                {"id": image_id, "file_name": file_name, "height": height, "width": width}\n            )\n            for cls, cx, cy, bw, bh in transform_rows(rows, geometry):\n                box_width = bw * width\n                box_height = bh * height\n                x = cx * width - box_width / 2\n                y = cy * height - box_height / 2\n                annotations.append(\n                    {\n                        "id": len(annotations) + 1,\n                        "image_id": image_id,\n                        "category_id": int(cls),\n                        "bbox": [float(x), float(y), float(box_width), float(box_height)],\n                        "area": float(box_width * box_height),\n                        "iscrowd": 0,\n                    }\n                )\n        payload = {\n            "info": {"description": "HSI 16-band fixed fold architecture screen"},\n            "images": images,\n            "annotations": annotations,\n            "categories": categories,\n        }\n        annotation_path = annotations_dir / f"instances_{split}.json"\n        annotation_path.write_text(json.dumps(payload, separators=(",", ":")))\n        evidence[f"{split}_annotations"] = len(annotations)\n        evidence[f"{split}_annotation_sha256"] = sha256(annotation_path)\n\n    evidence_path = output / "dataset_evidence.json"\n    evidence_path.write_text(json.dumps(evidence, indent=2) + "\\n")\n    return evidence\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--cube-dir", required=True)\n    parser.add_argument("--xml-dir", required=True)\n    parser.add_argument("--class-file", required=True)\n    parser.add_argument("--split-manifest", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    print(json.dumps(build_dataset(args), indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
import subprocess
import sys
from pathlib import Path

base_matches = list(Path('/content/kinput').rglob('hyperspectral-2026'))
legacy_matches = list(Path('/content/kinput').rglob('hsi_plan_c_ratio/split_manifest.json'))
assert len(base_matches) == 1, base_matches
assert len(legacy_matches) == 1, legacy_matches
base = base_matches[0]
fold_dir = Path('/kaggle/working/folds')
subprocess.run([
    sys.executable, '/kaggle/working/build_folds.py',
    '--xml-dir', str(base / 'data_train/data_train/Annotations/VIS'),
    '--cube-dir', str(base / 'data_train/data_train/VIS'),
    '--class-file', str(base / 'class.txt'),
    '--legacy-split', str(legacy_matches[0]),
    '--output-dir', str(fold_dir),
    '--seed', '42',
], check=True)
subprocess.run([
    sys.executable, '/kaggle/working/architecture_dataset.py',
    '--cube-dir', str(base / 'data_train/data_train/VIS'),
    '--xml-dir', str(base / 'data_train/data_train/Annotations/VIS'),
    '--class-file', str(base / 'class.txt'),
    '--split-manifest', str(fold_dir / 'fold0.json'),
    '--output-dir', '/kaggle/working/hsi_arch_data',
    '--seed', '42',
], check=True)


In [ ]:
import json, hashlib
from pathlib import Path
import numpy as np, torch

from mmengine.config import Config
from mmengine.runner import Runner
from mmcv.transforms import BaseTransform
from mmdet.registry import TRANSFORMS

# --- read the 18 class names (build a metainfo) ---
base = list(Path('/content/kinput').rglob('hyperspectral-2026'))[0]
classes = [c.strip() for c in (base/'class.txt').read_text().splitlines() if c.strip()]
assert len(classes) == 18, len(classes)

# --- custom transform: load a 16-band .npy as the image (float32, already 0..255 range) ---
@TRANSFORMS.register_module()
class LoadHSINpy(BaseTransform):
    def transform(self, results):
        arr = np.load(results['img_path']).astype('float32')     # H x W x 16
        results['img'] = arr
        results['img_shape'] = arr.shape[:2]
        results['ori_shape'] = arr.shape[:2]
        return results

data_root = '/kaggle/working/hsi_arch_data'   # produced by architecture_dataset.py (COCO json + images/{train,val}/*.npy)
cfg = Config.fromfile('/kaggle/working/repos/mmdetection/configs/cascade_rcnn/cascade-rcnn_r50_fpn_1x_coco.py')

# 16-band input stem + 16-channel normalization (npy already 0..255 -> scale to ~0..1)
cfg.model.backbone.in_channels = 16
cfg.model.data_preprocessor = dict(type='DetDataPreprocessor', mean=[0.0]*16, std=[255.0]*16,
                                   bgr_to_rgb=False, pad_size_divisor=32)
# 18 classes in every cascade bbox head
for head in cfg.model.roi_head.bbox_head:
    head.num_classes = 18
# do not try to load the 3-channel COCO conv1 into a 16-channel stem
cfg.model.backbone.init_cfg = None
cfg.load_from = None

metainfo = dict(classes=tuple(classes))
train_pipeline = [
    dict(type='LoadHSINpy'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='Resize', scale=(640, 640), keep_ratio=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadHSINpy'),
    dict(type='Resize', scale=(640, 640), keep_ratio=True),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='PackDetInputs', meta_keys=('img_id','img_path','ori_shape','img_shape','scale_factor')),
]
cfg.train_dataloader.batch_size = 2
cfg.train_dataloader.num_workers = 2
cfg.train_dataloader.dataset.update(dict(data_root=data_root, metainfo=metainfo,
    ann_file='annotations/instances_train.json', data_prefix=dict(img='images/train/'),
    pipeline=train_pipeline))
cfg.val_dataloader.dataset.update(dict(data_root=data_root, metainfo=metainfo,
    ann_file='annotations/instances_val.json', data_prefix=dict(img='images/val/'),
    pipeline=test_pipeline))
cfg.test_dataloader = cfg.val_dataloader
cfg.val_evaluator.update(dict(ann_file=f'{data_root}/annotations/instances_val.json', metric='bbox'))
cfg.test_evaluator = cfg.val_evaluator

cfg.train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=10, val_interval=1)
cfg.optim_wrapper.optimizer.update(dict(lr=0.005))
cfg.work_dir = '/kaggle/working/cr1-cascade-rcnn-f0'
cfg.default_hooks.checkpoint = dict(type='CheckpointHook', interval=10, save_best='coco/bbox_mAP')
cfg.randomness = dict(seed=42)

runner = Runner.from_cfg(cfg)
runner.train()
metrics = runner.val()
record = {
    'experiment_id': 'cr1-cascade-rcnn-f0', 'fold': 0, 'epochs': 10,
    'input_channels': 16, 'classes': 18, 'framework': 'mmdetection', 'model': 'cascade-rcnn_r50_fpn',
    'stem_initialization': 'random_16band', 'metrics': {k: float(v) for k, v in metrics.items()},
    'submission_created': False,
}
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
Path(f'{cfg.work_dir}/architecture_evidence.json').write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
